# Module 02 — Tensor Operations, Indexing & Broadcasting

**Prerequisites:** Module 01 (What Is PyTorch, and Your First Tensors)
**Time:** ~90 minutes

## Learning Objectives

- Perform arithmetic, matrix multiplication, and reduction operations on tensors.
- Index and slice tensors to extract specific elements, rows, or sub-tensors.
- Explain **broadcasting**: how PyTorch combines tensors of different (but compatible) shapes.
- Concatenate and stack tensors using `torch.cat` and `torch.stack`.
- Reorder tensor dimensions with `permute` and `transpose`.
- Use boolean masking and `torch.where` for conditional operations.
- Predict the output shape of an operation before running it.


In [ ]:
import torch


## Element-wise Operations

"Element-wise" means: apply the operation to each pair of matching positions independently. If two tensors have the same shape, `+`, `-`, `*`, `/` all work position by position.


In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])

print(a + b)   # [11, 22, 33]
print(a * b)   # [10, 40, 90]  -- NOT matrix multiplication, just position-by-position
print(b / a)   # [10, 10, 10]


## Matrix Multiplication vs. Element-wise Multiplication

This is one of the most important distinctions in the whole course, because getting it wrong produces a tensor that *runs* without error but is mathematically meaningless.

- `a * b` — **element-wise**: multiplies matching positions. Requires identical (or broadcastable) shapes.
- `a @ b` (or `torch.matmul(a, b)`) — **matrix multiplication**: the standard linear-algebra operation. Requires the *inner* dimensions to match: `(m, n) @ (n, p) -> (m, p)`.


In [ ]:
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])   # shape (2, 2)
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])   # shape (2, 2)

print("Element-wise (A * B):")
print(A * B)

print("\nMatrix multiplication (A @ B):")
print(A @ B)


### 🔮 Predict before you run

`A` has shape `(2, 3)` and `B` has shape `(3, 4)`. What shape will `A @ B` produce? What shape would `A @ B.T` produce (`.T` transposes the last two dimensions)?


In [ ]:
A = torch.randn(2, 3)
B = torch.randn(3, 4)

result = A @ B
print("A @ B shape:", result.shape)   # inner dims (3, 3) match and cancel out

try:
    bad = A @ B.T   # (2,3) @ (4,3) -- inner dims 3 and 4 don't match
except RuntimeError as e:
    print("\nA @ B.T fails:")
    print(e)


**Rule to internalize:** for `X @ Y`, the *last* dimension of `X` must equal the *second-to-last* dimension of `Y`. This single rule explains nearly every "size mismatch" error you'll hit once you start building real layers (`nn.Linear` is matrix multiplication under the hood, as you'll see in Module 04).


## Common Mathematical Operations

These appear frequently in loss functions, activations, and data preprocessing.


In [ ]:
x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])

print("abs:  ", torch.abs(x))           # absolute value
print("clamp:", torch.clamp(x, -2, 2))  # clip values to [-2, 2]
print("exp:  ", torch.exp(x))           # e^x (used in softmax)
print("log:  ", torch.log(torch.abs(x) + 1e-8))  # natural log (add epsilon to avoid log(0))
print("sqrt: ", torch.sqrt(torch.abs(x)))         # square root
print("pow:  ", torch.pow(x, 2))        # x^2 — same as x ** 2


**`torch.clamp`** is particularly useful for gradient clipping and ensuring numerical stability — you'll see it in advanced training loops.

**`torch.exp`** is the core of the softmax function, and **`torch.log`** appears in cross-entropy loss calculations.


## Broadcasting

Broadcasting lets PyTorch operate on two tensors of *different* shapes, by automatically "stretching" the smaller one — without actually copying data in memory — so the shapes line up.

**The broadcasting rule (compare shapes right-to-left):**
1. Align the shapes from the rightmost dimension.
2. Two dimensions are compatible if they're equal, **or** one of them is 1.
3. Missing dimensions on the left are treated as size 1.

If every pair of dimensions satisfies rule 2, the operation succeeds; the size-1 dimensions get conceptually "repeated" to match.


In [ ]:
# Adding a scalar to every element -- the simplest broadcasting case
x = torch.tensor([1.0, 2.0, 3.0])
print(x + 10)   # 10 is broadcast to shape (3,) -> [11, 12, 13]

# Adding a (3,) vector to each row of a (2, 3) matrix
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])   # shape (2, 3)
row = torch.tensor([10.0, 20.0, 30.0])                       # shape (3,)
print(matrix + row)
# row is treated as if it were shape (1, 3), then "stretched" to (2, 3)


Walking through the rule for `matrix (2, 3) + row (3,)`:

```
matrix: (2, 3)
row:       (3,)   -> treated as (1, 3) since it's missing a leading dimension
```
Compare right-to-left: `3 == 3` ✓, and `2` vs `1` → compatible because one of them is 1 ✓. Result shape: `(2, 3)`.


In [ ]:
# Advanced: outer product via broadcasting
# Two vectors, reshaped so they broadcast into a matrix
a = torch.tensor([1.0, 2.0, 3.0]).reshape(3, 1)   # column vector (3, 1)
b = torch.tensor([10.0, 20.0])                      # row vector (2,) -> treated as (1, 2)

# (3, 1) * (1, 2) -> (3, 2)
outer = a * b
print("outer product shape:", outer.shape)
print(outer)


In [ ]:
# Practical: per-channel normalization of an image batch
# Image batch: (batch=8, channels=3, height=32, width=32)
images = torch.randn(8, 3, 32, 32)

# Per-channel mean: shape (3,) -> reshape to (1, 3, 1, 1) for broadcasting
channel_mean = torch.tensor([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
channel_std  = torch.tensor([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)

# These are the ImageNet normalization constants — you'll use this exact pattern later
normalized = (images - channel_mean) / channel_std
print("normalized shape:", normalized.shape)   # still (8, 3, 32, 32)


### 🐛 Debugging Challenge: Broken Broadcast

The cell below is intended to add a per-column bias vector to every row of a batch of feature vectors. It has a bug. Can you spot it before running?


In [ ]:
batch = torch.randn(4, 3)      # 4 samples, 3 features each
bias = torch.randn(4)           # WRONG shape -- intended to be one value per feature (3), not per sample (4)

try:
    result = batch + bias
except RuntimeError as e:
    print("Error:", e)


**Diagnosis:** `bias` has shape `(4,)` but the last dimension of `batch` is `3`. Comparing right-to-left: `3` vs `4` — neither equals the other, and neither is `1`, so broadcasting fails.

**Fix:** `bias` should have shape `(3,)` — one value per feature/column, matching `batch`'s last dimension.


In [ ]:
bias_fixed = torch.randn(3)     # correct: one bias value per feature
result = batch + bias_fixed
print(result.shape)   # (4, 3) -- bias_fixed broadcasts across all 4 rows


## Indexing and Slicing

Tensor indexing follows the same `[start:stop:step]` conventions as Python lists and NumPy, extended across multiple dimensions.


In [ ]:
x = torch.arange(20).reshape(4, 5)
print(x)
print()

print("row 0:          ", x[0])
print("column 2:        ", x[:, 2])          # all rows, column index 2
print("rows 1-2:       \n", x[1:3])          # rows with index 1 and 2 (3 excluded)
print("single element: ", x[2, 3].item())     # .item() extracts a plain Python number
print("last row:       ", x[-1])


`.item()` converts a single-element tensor into an ordinary Python number (`int` or `float`). You'll use this constantly when printing loss values or accuracy during training — `loss.item()` appears in nearly every training loop you'll ever write.


## Boolean Masking and `torch.where`

Boolean masking lets you select elements based on a condition — essential for filtering, thresholding, and attention masking.


In [ ]:
x = torch.tensor([1.0, -2.0, 3.0, -4.0, 5.0])

# Create a boolean mask
mask = x > 0
print("mask:", mask)           # tensor([True, False, True, False, True])

# Use it to select elements
positives = x[mask]
print("positives:", positives)  # tensor([1., 3., 5.])

# Count elements matching a condition
print("number of positives:", mask.sum().item())


In [ ]:
# torch.where: conditional selection — like an element-wise if/else
x = torch.tensor([-1.0, 2.0, -3.0, 4.0])

# Replace negatives with 0 (this is what ReLU does!)
result = torch.where(x > 0, x, torch.tensor(0.0))
print("ReLU-like:", result)

# Practical: clamp predictions to [0, 1]
predictions = torch.tensor([-0.5, 0.3, 1.2, 0.8])
clamped = torch.clamp(predictions, 0.0, 1.0)   # equivalent but cleaner
print("clamped:", clamped)


## Reduction Operations

Reductions collapse a tensor along one or more dimensions, producing a smaller tensor. The `dim` argument controls *which* dimension gets collapsed.


In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])   # shape (2, 3)

print("sum, no dim (collapses everything):", x.sum())
print("sum over dim=0 (collapse rows, keep 3 columns):", x.sum(dim=0))
print("sum over dim=1 (collapse columns, keep 2 rows):", x.sum(dim=1))
print("mean over dim=1:", x.mean(dim=1))
print("max over dim=1:", x.max(dim=1))   # returns both values AND their indices


**Mental shortcut for `dim`:** `dim=0` operates "down the rows" (collapsing across rows, one result per column); `dim=1` operates "across the columns" (collapsing across columns, one result per row). If you're ever unsure, the safest approach is to try it and check `.shape` — which is exactly the habit this course wants you to build.


In [ ]:
# argmax — returns the INDEX of the maximum value
# This is how you convert model logits -> predicted class in classification
logits = torch.tensor([[2.0, 1.0, 0.1],
                       [0.1, 3.0, 0.2],
                       [0.5, 0.5, 4.0]])

predicted_classes = logits.argmax(dim=1)   # index of max in each row
print("predicted classes:", predicted_classes)  # [0, 1, 2]


## `torch.cat` and `torch.stack`

These are the two main ways to combine multiple tensors:

- **`torch.cat`** — **concatenate** along an *existing* dimension. Like gluing tensors end-to-end.
- **`torch.stack`** — **stack** along a *new* dimension. Like stacking plates into a pile.


In [ ]:
a = torch.tensor([[1, 2], [3, 4]])   # shape (2, 2)
b = torch.tensor([[5, 6], [7, 8]])   # shape (2, 2)

# cat along dim=0: rows are joined
cat_rows = torch.cat([a, b], dim=0)
print("cat dim=0 (join rows):")
print(cat_rows)
print("shape:", cat_rows.shape)   # (4, 2)

# cat along dim=1: columns are joined
cat_cols = torch.cat([a, b], dim=1)
print("\ncat dim=1 (join columns):")
print(cat_cols)
print("shape:", cat_cols.shape)   # (2, 4)


In [ ]:
# stack creates a NEW dimension
stacked = torch.stack([a, b], dim=0)
print("stack dim=0:")
print(stacked)
print("shape:", stacked.shape)   # (2, 2, 2) — a new dimension was inserted at position 0

stacked_dim1 = torch.stack([a, b], dim=1)
print("\nstack dim=1:")
print(stacked_dim1)
print("shape:", stacked_dim1.shape)   # (2, 2, 2) — new dim at position 1


### 🔮 Quick quiz: `cat` vs `stack`

Given `a` with shape `(3, 4)` and `b` with shape `(3, 4)`:

- `torch.cat([a, b], dim=0)` → shape `(6, 4)` (joined 3+3 rows)
- `torch.cat([a, b], dim=1)` → shape `(3, 8)` (joined 4+4 columns)
- `torch.stack([a, b], dim=0)` → shape `(2, 3, 4)` (new dimension of size 2)

**Common use cases:**
- `torch.cat` — combining predictions from multiple batches, concatenating features
- `torch.stack` — collecting per-epoch losses into a single tensor, creating batches from individual samples


## `permute` and `transpose`

These reorder dimensions without changing the data. Critical for converting between different frameworks' conventions (e.g., channel-first vs channel-last images).


In [ ]:
# An image in NumPy/TensorFlow format: (height, width, channels)
image_hwc = torch.randn(224, 224, 3)
print("HWC format:", image_hwc.shape)

# Convert to PyTorch format: (channels, height, width)
image_chw = image_hwc.permute(2, 0, 1)   # move dim 2 to front, then dims 0, 1
print("CHW format:", image_chw.shape)

# transpose: swap exactly two dimensions
matrix = torch.randn(3, 5)
transposed = matrix.transpose(0, 1)   # swap rows and columns
print("\noriginal:", matrix.shape, "-> transposed:", transposed.shape)
print("same as .T:", matrix.T.shape)   # shorthand for 2D tensors


In [ ]:
# Practical: convert a batch of images from (B, H, W, C) to (B, C, H, W)
batch_hwc = torch.randn(8, 32, 32, 3)   # e.g., loaded from a NumPy array
batch_chw = batch_hwc.permute(0, 3, 1, 2)   # keep batch dim, move channel to pos 1
print("before:", batch_hwc.shape)
print("after:", batch_chw.shape)


## In-place Operations (Revisited)

Recall from Module 01: any operation with a trailing underscore (`_`) modifies the tensor in place. Here are the most common ones you'll encounter in practice:


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])

# In-place versions
x.add_(10)        # x = x + 10
print("add_(10):", x)

x.clamp_(0, 15)   # x = clamp(x, 0, 15)
print("clamp_(0, 15):", x)

x.fill_(0)        # set ALL elements to 0
print("fill_(0):", x)

# The most important in-place op: zero_()
# This is what optimizer.zero_grad() calls internally on every gradient
grad = torch.tensor([0.5, -0.3, 0.8])
print("\nbefore zero_():", grad)
grad.zero_()
print("after zero_():", grad)


## 🧠 Shape Prediction Quiz

Without running any code, predict the output shape for each operation. Then check your answers in the cell below.

1. `torch.randn(3, 4) @ torch.randn(4, 5)` → ?
2. `torch.randn(3, 1) + torch.randn(1, 4)` → ?
3. `torch.randn(8, 3, 32, 32).sum(dim=1)` → ?
4. `torch.cat([torch.randn(5, 3), torch.randn(7, 3)], dim=0)` → ?
5. `torch.stack([torch.randn(3, 4), torch.randn(3, 4)], dim=0)` → ?
6. `torch.randn(2, 3, 4).permute(2, 0, 1)` → ?
7. `torch.randn(5).unsqueeze(0).unsqueeze(-1)` → ?
8. `torch.randn(1, 3, 1) * torch.randn(4, 1, 5)` → ?
9. `torch.randn(10, 3).argmax(dim=1)` → ?
10. `torch.randn(6, 4).reshape(2, 3, -1)` → ?


In [ ]:
# Answers — run to verify
print("1. matmul:", (torch.randn(3, 4) @ torch.randn(4, 5)).shape)       # (3, 5)
print("2. broadcast:", (torch.randn(3, 1) + torch.randn(1, 4)).shape)    # (3, 4)
print("3. sum dim=1:", torch.randn(8, 3, 32, 32).sum(dim=1).shape)       # (8, 32, 32)
print("4. cat:", torch.cat([torch.randn(5, 3), torch.randn(7, 3)], dim=0).shape)  # (12, 3)
print("5. stack:", torch.stack([torch.randn(3, 4), torch.randn(3, 4)], dim=0).shape)  # (2, 3, 4)
print("6. permute:", torch.randn(2, 3, 4).permute(2, 0, 1).shape)        # (4, 2, 3)
print("7. unsqueeze:", torch.randn(5).unsqueeze(0).unsqueeze(-1).shape)   # (1, 5, 1)
print("8. broadcast:", (torch.randn(1, 3, 1) * torch.randn(4, 1, 5)).shape)  # (4, 3, 5)
print("9. argmax:", torch.randn(10, 3).argmax(dim=1).shape)               # (10,)
print("10. reshape:", torch.randn(6, 4).reshape(2, 3, -1).shape)          # (2, 3, 4)


## Exercises

🟢 **Beginner 1:** Create two tensors of shape `(3,)`, add them, multiply them element-wise, and compute the dot product using `torch.dot`.

🟢 **Beginner 2:** Create a tensor `x = torch.arange(20).reshape(4, 5)`. Extract: row 2, column 3, the sub-matrix containing rows 1-2 and columns 2-4.

🟡 **Intermediate 1:** Create a tensor `x` of shape `(5, 4)` using `torch.randn`. Compute the mean of each row (result shape `(5,)`) and the mean of each column (result shape `(4,)`) using `dim`.

🟡 **Intermediate 2:** Create three tensors of shape `(2, 3)`. Concatenate them along dim=0 and verify the result has shape `(6, 3)`. Stack them along dim=0 and verify the result has shape `(3, 2, 3)`.

🟡 **Intermediate 3:** Create a tensor `x = torch.randn(10)`. Use boolean masking to extract all values greater than 0.5. Count how many there are.

🔴 **Challenge 1:** You have `images` of shape `(batch=8, channels=3, height=32, width=32)` and want to subtract a per-channel mean `mean = torch.tensor([0.5, 0.4, 0.3])` (shape `(3,)`) from every pixel in every image. Reshape `mean` so that broadcasting works, then apply the subtraction.

🔴 **Challenge 2:** Create a tensor of "model logits" with shape `(16, 5)` (16 samples, 5 classes). Use `argmax` to get predicted classes. Create random "true labels" with `torch.randint(0, 5, (16,))`. Compute accuracy as the fraction of matching predictions.


In [ ]:
# Space for your exercise solutions



## Common Mistakes

- **Using `*` when you meant `@`** (or vice versa). Element-wise and matrix multiplication produce *different shapes and different numbers*, and `*` will often run without error even when it's semantically wrong — always sanity-check output shapes.
- **Assuming broadcasting "just works."** It only works when shapes are compatible right-to-left. When in doubt, print both `.shape`s before the operation.
- **Confusing `torch.cat` with `torch.stack`** — `cat` joins along an existing dimension (same ndim), `stack` creates a new one (ndim + 1).
- **Forgetting `dim` matters** for reductions — `sum(dim=0)` and `sum(dim=1)` give different-shaped, different results.
- **Using `.item()` on a multi-element tensor** — `.item()` only works on tensors with exactly one element. For multiple values, use `.tolist()` instead.

## Mental Model

Think of broadcasting as PyTorch asking, for every dimension from the right: *"are these the same size, or is one of them just '1', meaning 'use this same value everywhere'?"* If yes to every dimension, it proceeds; if any dimension disagrees, it stops with an error rather than guessing.

## Key Takeaways

- `*` is element-wise; `@` is matrix multiplication — never assume, always check.
- Broadcasting compares shapes right-to-left; a dimension of size 1 (or a missing dimension) is stretched to match.
- `torch.cat` joins along an existing dimension; `torch.stack` creates a new one.
- `permute`/`transpose` reorder dimensions — essential for converting between `(B,H,W,C)` and `(B,C,H,W)`.
- Boolean masking and `torch.where` enable conditional element selection.
- `dim` controls which axis a reduction collapses — always double-check the resulting shape.
- Indexing and slicing work like NumPy; `.item()` converts a single-value tensor to a plain Python number.

## What's Next

**Module 03 — Autograd and Computational Graphs** introduces the second pillar of PyTorch: automatic differentiation. You now have the tensor mechanics needed to understand what autograd is actually tracking.

## Checklist

- [ ] I can distinguish element-wise multiplication from matrix multiplication and predict which one a given line of code performs
- [ ] I can explain the broadcasting rule in my own words
- [ ] I can predict the output shape of a broadcasted operation before running it
- [ ] I know the difference between `torch.cat` and `torch.stack`
- [ ] I can use `permute` to convert between channel-first and channel-last image formats
- [ ] I can use boolean masking and `torch.where` for conditional operations
- [ ] I can use `dim` correctly in a reduction and explain what it collapses
- [ ] I can predict the output shape for at least 8/10 questions in the shape quiz
